# Notebook 05 — Residual Correction with Gradient-Boosted Trees (Research Plan v2)

Implements `docs/RESEARCH_PLAN_V2.md`: a **well-posed, CPU-only** re-do of the SGP4
error-correction study after NB01–04 returned a negative result.

Key differences from NB01–04:
1. **Real ground truth** — each TLE's error is measured against the *next* independently
   fitted TLE (not against another SGP4 run of itself).
2. **RIC-frame target** (radial / along-track / cross-track, km) — no geodetic-degree pathologies.
3. **Space-weather features** (F10.7, Ap, Kp) — the physical driver of drag error SGP4 misses.
4. **LightGBM**, one model per RIC axis — trains in seconds on CPU, no GPU / Colab needed.

> **Status: scaffold.** The cells are written to run top-to-bottom on your machine
> (Space-Track creds in `../.env`) or Colab. Fill in `docs/RESEARCH_PLAN_V2.md` §5 tables
> from the evaluation cell output. Nothing here is executed yet.


## 0 — Environment setup

In [ ]:
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'sgp4', 'lightgbm', 'scikit-learn',
                    'python-dotenv', 'requests', 'pandas', 'plotly'], check=True)
else:
    # lightgbm is research-only (not in requirements.txt); install locally if missing
    try:
        import lightgbm  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)

import os, math, time
from pathlib import Path
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import requests
from sgp4.api import Satrec, jday

DATA_DIR = Path('/content') if IN_COLAB else Path('../data/collected')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK. Data dir:', DATA_DIR.resolve())

## 1 — Configuration

In [ ]:
NORAD_IDS = [43017, 43137, 25544, 33591, 20580]   # same LEO set as NB01
HISTORY_DAYS = 180
HORIZONS_H   = [1, 3, 6, 12, 24, 48, 72]   # hours since prediction-TLE epoch
TRUTH_TOL_H  = 36.0    # max |epoch(truth TLE) - t_eval| to accept a truth sample (hours)
MAX_ERR_KM   = 500.0   # drop decayed / bad TLE pairs (as in the shipped model)

BASE_URL = 'https://www.space-track.org'
SW_URL   = 'https://celestrak.org/SpaceData/SW-Last5Years.csv'   # free, no account
print(f'{len(NORAD_IDS)} satellites, horizons {HORIZONS_H} h')

## 2 — Credentials (Space-Track)

Local: `../.env` is loaded automatically. Colab: upload `.env` or set Colab Secrets.

In [ ]:
if not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=Path('../.env'), override=False)
    except ImportError:
        pass

ST_USER = os.environ.get('SPACETRACK_USER', '')
ST_PASS = os.environ.get('SPACETRACK_PASS', '')
if not ST_USER or not ST_PASS:
    raise ValueError('Space-Track credentials not found (SPACETRACK_USER / SPACETRACK_PASS).')
print('Credentials loaded for:', ST_USER)

## 3 — Fetch TLE history (reuses NB01 logic)

In [ ]:
def login_spacetrack(session):
    r = session.post(f'{BASE_URL}/ajaxauth/login',
                     data={'identity': ST_USER, 'password': ST_PASS})
    r.raise_for_status()
    if 'Failed' in r.text:
        raise RuntimeError('Space-Track login failed.')

def fetch_tle_history(session, norad_id, days):
    since = (datetime.now(timezone.utc) - timedelta(days=days)).strftime('%Y-%m-%d')
    url = (f'{BASE_URL}/basicspacedata/query/class/gp_history'
           f'/NORAD_CAT_ID/{norad_id}/orderby/EPOCH asc'
           f'/EPOCH/{since}--now/format/tle')
    r = session.get(url); r.raise_for_status()
    lines = [l.strip() for l in r.text.strip().splitlines() if l.strip()]
    pairs = [(lines[i], lines[i+1]) for i in range(0, len(lines)-1, 2)
             if lines[i].startswith('1') and lines[i+1].startswith('2')]
    return pairs

session = requests.Session()
login_spacetrack(session)
tle_history = {}
for nid in NORAD_IDS:
    time.sleep(1)
    tle_history[nid] = fetch_tle_history(session, nid, HISTORY_DAYS)
    print(f'  NORAD {nid}: {len(tle_history[nid])} TLEs')
session.close()

## 4 — Fetch space weather (CelesTrak, free)

F10.7 solar flux and Ap/Kp geomagnetic indices are the standard NRLMSISE-00 drag drivers —
the physical cause of the along-track error SGP4 accumulates. One small CSV, joined on date.

In [ ]:
sw_raw = pd.read_csv(SW_URL)
sw_raw.columns = [c.strip().upper() for c in sw_raw.columns]

def _pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series(np.nan, index=df.index)

sw = pd.DataFrame({'date': pd.to_datetime(_pick(sw_raw, 'DATE')).dt.date})
sw['f107']     = pd.to_numeric(_pick(sw_raw, 'F10.7_OBS', 'F10.7_ADJ'), errors='coerce')
sw['f107_81d'] = pd.to_numeric(_pick(sw_raw, 'F10.7_OBS_CENTER81', 'F10.7_OBS_LAST81'), errors='coerce')
sw['ap']       = pd.to_numeric(_pick(sw_raw, 'AP_AVG'), errors='coerce')
sw['kp_sum']   = pd.to_numeric(_pick(sw_raw, 'KP_SUM'), errors='coerce')
# 1-day lag captures storm after-effects on density
sw = sw.sort_values('date').reset_index(drop=True)
for col in ['f107', 'ap', 'kp_sum']:
    sw[f'{col}_lag1'] = sw[col].shift(1)
sw_by_date = sw.set_index('date')
print('Space weather rows:', len(sw), '| range:', sw['date'].min(), '->', sw['date'].max())
sw.tail(3)

## 5 — Helpers: epoch parsing, orbital features, RIC transform

In [ ]:
def tle_epoch_dt(line1):
    s = line1[18:32].strip()
    yy = int(s[:2]); year = 2000 + yy if yy < 57 else 1900 + yy
    return datetime(year, 1, 1, tzinfo=timezone.utc) + timedelta(days=float(s[2:]) - 1)

MU = 398600.4418  # km^3/s^2

def orbital_features(sat):
    n_rad_min = sat.no_kozai                      # rad/min
    mean_motion = n_rad_min * 1440.0 / (2*math.pi)  # rev/day
    n_rad_s = n_rad_min / 60.0
    altitude = (MU / n_rad_s**2) ** (1/3) - 6378.137
    return {
        'mean_motion': mean_motion, 'eccentricity': sat.ecco,
        'inclination_deg': math.degrees(sat.inclo), 'raan_deg': math.degrees(sat.nodeo),
        'arg_perigee_deg': math.degrees(sat.argpo), 'bstar': sat.bstar,
        'altitude_km': altitude,
    }

def state_at(sat, t_dt):
    jd, fr = jday(t_dt.year, t_dt.month, t_dt.day, t_dt.hour, t_dt.minute,
                  t_dt.second + t_dt.microsecond/1e6)
    e, r, v = sat.sgp4(jd, fr)
    if e != 0:
        return None, None
    return np.array(r), np.array(v)   # TEME km, km/s

def ric_components(r_pred, v_pred, err_vec):
    """Project an ECI/TEME error vector onto radial / in-track / cross-track axes (km)."""
    R = r_pred / np.linalg.norm(r_pred)
    C = np.cross(r_pred, v_pred); C = C / np.linalg.norm(C)
    I = np.cross(C, R)
    return np.array([err_vec @ R, err_vec @ I, err_vec @ C])   # [radial, in-track, cross-track]

## 6 — Build the sample table

For each prediction TLE and each horizon Δ, truth = the TLE whose epoch is closest to
`t_eval = epoch + Δ` (within `TRUTH_TOL_H`). The real SGP4 error is
`SGP4(truth)@t_eval − SGP4(pred)@t_eval`, expressed in the RIC frame.

In [ ]:
def nearest_truth(sat_list, t_eval, pred_epoch):
    """Pick the TLE with epoch nearest t_eval but different from the prediction TLE."""
    best, best_dt = None, None
    for ep, sat in sat_list:
        if abs((ep - pred_epoch).total_seconds()) < 60:   # skip the prediction TLE itself
            continue
        dt = abs((ep - t_eval).total_seconds()) / 3600.0
        if best_dt is None or dt < best_dt:
            best, best_dt = (ep, sat), dt
    return best, (best_dt if best_dt is not None else 1e9)

rows = []
for nid, pairs in tle_history.items():
    sat_list = sorted(((tle_epoch_dt(l1), Satrec.twoline2rv(l1, l2)) for l1, l2 in pairs),
                      key=lambda x: x[0])
    for pred_epoch, pred_sat in sat_list:
        feats0 = orbital_features(pred_sat)
        for dh in HORIZONS_H:
            t_eval = pred_epoch + timedelta(hours=dh)
            truth, gap_h = nearest_truth(sat_list, t_eval, pred_epoch)
            if truth is None or gap_h > TRUTH_TOL_H:
                continue
            r_pred, v_pred = state_at(pred_sat, t_eval)
            r_true, _      = state_at(truth[1], t_eval)
            if r_pred is None or r_true is None:
                continue
            err_ric = ric_components(r_pred, v_pred, r_true - r_pred)
            if np.linalg.norm(err_ric) > MAX_ERR_KM:
                continue
            row = dict(feats0)
            row.update(norad_id=nid, t_eval=t_eval, time_since_epoch_h=float(dh),
                       err_radial=err_ric[0], err_intrack=err_ric[1], err_crosstrack=err_ric[2])
            rows.append(row)

samples = pd.DataFrame(rows).sort_values('t_eval').reset_index(drop=True)
print(f'Built {len(samples):,} samples')
print(samples[['err_radial','err_intrack','err_crosstrack']].describe().round(2))

## 7 — Join space weather & assemble feature matrix

In [ ]:
samples['date'] = pd.to_datetime(samples['t_eval']).dt.date
sw_cols = ['f107', 'f107_81d', 'ap', 'kp_sum', 'f107_lag1', 'ap_lag1', 'kp_sum_lag1']
samples = samples.join(sw_by_date[sw_cols], on='date')
samples[sw_cols] = samples[sw_cols].ffill().bfill()   # fill any date gaps

ORBIT_FEATS = ['mean_motion','eccentricity','inclination_deg','raan_deg',
               'arg_perigee_deg','bstar','altitude_km','time_since_epoch_h']
SW_FEATS    = sw_cols
TARGETS     = ['err_radial','err_intrack','err_crosstrack']
print('Orbit feats:', len(ORBIT_FEATS), '| SW feats:', len(SW_FEATS))
samples.head(3)

## 8 — Time-ordered split & LightGBM training

Baseline to beat = **raw SGP4** (predict zero correction → residual equals the target itself).
One LightGBM model per RIC axis; ablation compares orbit-only vs orbit+space-weather.

In [ ]:
import lightgbm as lgb

n = len(samples); n_tr = int(n*0.70); n_va = int(n*0.15)
train, val, test = samples.iloc[:n_tr], samples.iloc[n_tr:n_tr+n_va], samples.iloc[n_tr+n_va:]
print(f'Train {len(train):,}  Val {len(val):,}  Test {len(test):,}')

def train_axis(feat_cols, target):
    m = lgb.LGBMRegressor(objective='huber', alpha=0.9, n_estimators=600,
                          learning_rate=0.03, num_leaves=31, subsample=0.8,
                          colsample_bytree=0.8, min_child_samples=40, verbosity=-1)
    m.fit(train[feat_cols], train[target],
          eval_set=[(val[feat_cols], val[target])], eval_metric='l1',
          callbacks=[lgb.early_stopping(40, verbose=False)])
    return m

def fit_predict(feat_cols):
    preds = {}
    for tgt in TARGETS:
        preds[tgt] = train_axis(feat_cols, tgt).predict(test[feat_cols])
    return pd.DataFrame(preds, index=test.index)

pred_orbit    = fit_predict(ORBIT_FEATS)
pred_orbit_sw = fit_predict(ORBIT_FEATS + SW_FEATS)
print('Trained orbit-only and orbit+space-weather models.')

## 9 — Evaluation (fills RESEARCH_PLAN_V2 §5)

In [ ]:
def mae(a):  return float(np.mean(np.abs(a)))

def report(name, pred_df):
    out = {}
    for tgt in TARGETS:
        base = mae(test[tgt])                      # raw SGP4 (no correction)
        corr = mae(test[tgt] - pred_df[tgt])       # after ML correction
        out[tgt] = (round(base,3), round(corr,3), round(100*(base-corr)/base,1))
    return out

for name, pdf in [('orbit-only', pred_orbit), ('orbit+SW', pred_orbit_sw)]:
    print(f'\n=== {name}:  axis -> (SGP4 MAE km, corrected MAE km, % improvement) ===')
    for k, v in report(name, pdf).items():
        print(f'  {k:14s} SGP4={v[0]:7.3f}  corrected={v[1]:7.3f}  improvement={v[2]:+5.1f}%')

# In-track (along-track) MAE by time-since-epoch cohort — the key staleness table
test_ = test.copy(); test_['corr_intrack'] = pred_orbit_sw['err_intrack']
test_['age_bin'] = pd.cut(test_['time_since_epoch_h']/24.0, [0,1,3,5,10],
                          labels=['<1d','1-3d','3-5d','>5d'])
print('\n=== In-track MAE (km) by TLE age — SGP4 vs corrected (orbit+SW) ===')
for b, g in test_.groupby('age_bin', observed=True):
    print(f'  {str(b):6s} n={len(g):6d}  SGP4={mae(g.err_intrack):7.3f}  '
          f'corrected={mae(g.err_intrack - g.corr_intrack):7.3f}')

## 10 — Interpretation & next step

- **If corrected MAE < SGP4 MAE** on the in-track axis (especially in the 1–3d / 3–5d cohorts):
  the approach works. Save the three LightGBM models and wire them into `src/ml/` behind the
  existing app toggle (replace the scalar corrector with the RIC-vector one).
- **If not:** record the numbers in `docs/RESEARCH_PLAN_V2.md` §5 as a second, better-posed
  negative result and stop — the design is now sound, so the conclusion is trustworthy.
- **Cheap wins to try before giving up:** add `beta_angle_deg`; widen `HISTORY_DAYS`;
  quantile objective for uncertainty; per-satellite vs pooled models.

No GPU was used anywhere in this notebook.
